# 📖 Notebook 1: Embeddings & Similarity Search

Before we can search for "similar" things, we need to understand how computers represent similarity.

## Learning Objectives

By the end of this notebook, you'll understand:
- What vector embeddings are and why they matter
- How similarity metrics (L2, cosine) work
- **BAD**: Why linear scan through all vectors is a disaster at scale
- **BETTER**: How IVFFlat indexes partition the search space
- **BEST**: How HNSW graphs deliver fast, high-recall search

## 🛠️ Setup

Start the infrastructure first:

```bash
cd deep-dives/vector-databases
docker-compose up -d
```

### Adminer (PostgreSQL GUI)
- **URL**: http://localhost:8081
- Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `vector_demo`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import numpy as np
import time

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "vector_demo",
    "user": "demo",
    "password": "demo"
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

# Test connection
try:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM movies")
    count = cur.fetchone()[0]
    conn.close()
    print(f"✅ Connected to PostgreSQL — {count} movies loaded")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 What Are Embeddings?

An **embedding** (or **vector**) is an array of numbers that represents something — a word, a sentence, a movie, an image. The key insight:

> **Similar things end up with similar vectors.**

```
"The Matrix"          → [0.12, -0.34, 0.78, ..., 0.45]   (128 numbers)
"Blade Runner 2049"   → [0.11, -0.32, 0.79, ..., 0.44]   (very similar! both Sci-Fi)

"The Big Lebowski"    → [-0.89, 0.12, -0.45, ..., 0.23]  (very different — Comedy)
```

In our database, each movie already has a 128-dimensional embedding vector. Movies in the same genre have similar embeddings because they were generated with genre-based clustering.

### How are embeddings created in real systems?

In production, you'd use a **pre-trained model** (like OpenAI's embedding API, Sentence Transformers, or CLIP for images). The model converts your data into fixed-length vectors. For this lab, we use synthetic embeddings that cluster by genre — simpler to set up, same concepts apply.

In [ ]:
# Let's look at what embeddings actually look like

conn = get_conn()
cur = conn.cursor()

# Fetch two Sci-Fi movies and one Comedy
cur.execute("""
    SELECT title, genre, embedding::text
    FROM movies
    WHERE title IN ('The Matrix', 'Blade Runner 2049', 'The Big Lebowski')
    ORDER BY title
""")

for title, genre, emb_text in cur.fetchall():
    # Parse the vector string "[0.1,0.2,...]" into a list of floats
    values = [float(x) for x in emb_text.strip('[]').split(',')]
    print(f"🎬 {title} ({genre})")
    print(f"   Dimensions: {len(values)}")
    print(f"   First 8 values: {[round(v, 3) for v in values[:8]]}")
    print()

conn.close()

print("💡 Notice: Same-genre movies have similar first values (same base pattern).")
print("   Different-genre movies have very different values.")

## 📏 Similarity Metrics

How do we measure "how similar" two vectors are? There are three common metrics:

### L2 (Euclidean) Distance — `<->`
Straight-line distance between two points. **Smaller = more similar.**
```
distance = sqrt( (a1-b1)² + (a2-b2)² + ... + (aN-bN)² )
```

### Cosine Distance — `<=>`
Measures the angle between two vectors (ignores magnitude). **Smaller = more similar.**
```
cosine_distance = 1 - (a · b) / (|a| × |b|)
```

### Inner Product — `<#>`
Dot product of two vectors. In pgvector, **smaller (more negative) = more similar** (it returns negative inner product).

**Which to use?**
- **Cosine distance** is the default for text embeddings (most robust)
- **L2 distance** is good when magnitude matters
- **Inner product** is fastest when vectors are already normalized

In [ ]:
# Let's see similarity metrics in action

conn = get_conn()
cur = conn.cursor()

print("📏 Distance Between Movies")
print("=" * 70)

# Compare: Sci-Fi vs Sci-Fi (should be small distance)
cur.execute("""
    SELECT
        a.title, b.title,
        a.embedding <-> b.embedding AS l2_distance,
        a.embedding <=> b.embedding AS cosine_distance
    FROM movies a, movies b
    WHERE a.title = 'The Matrix' AND b.title = 'Interstellar'
""")
row = cur.fetchone()
print(f"\n🎬 {row[0]} ↔ {row[1]}  (same genre: Sci-Fi)")
print(f"   L2 distance:     {row[2]:.4f}")
print(f"   Cosine distance: {row[3]:.4f}")

# Compare: Sci-Fi vs Comedy (should be large distance)
cur.execute("""
    SELECT
        a.title, b.title,
        a.embedding <-> b.embedding AS l2_distance,
        a.embedding <=> b.embedding AS cosine_distance
    FROM movies a, movies b
    WHERE a.title = 'The Matrix' AND b.title = 'The Big Lebowski'
""")
row = cur.fetchone()
print(f"\n🎬 {row[0]} ↔ {row[1]}  (different genres)")
print(f"   L2 distance:     {row[2]:.4f}")
print(f"   Cosine distance: {row[3]:.4f}")

# Compare: Sci-Fi vs Thriller (should be moderate)
cur.execute("""
    SELECT
        a.title, b.title,
        a.embedding <-> b.embedding AS l2_distance,
        a.embedding <=> b.embedding AS cosine_distance
    FROM movies a, movies b
    WHERE a.title = 'The Matrix' AND b.title = 'Se7en'
""")
row = cur.fetchone()
print(f"\n🎬 {row[0]} ↔ {row[1]}  (different but related genres)")
print(f"   L2 distance:     {row[2]:.4f}")
print(f"   Cosine distance: {row[3]:.4f}")

conn.close()

print("\n💡 Same-genre movies have SMALLER distances (more similar).")
print("   Different-genre movies have LARGER distances (less similar).")

## 🔍 Similarity Search: Finding the K Nearest Neighbors

The fundamental operation in vector databases: given a query vector, find the K most similar vectors.

This is called **K-Nearest Neighbors (KNN)**. Let's try it:

In [ ]:
# Find the 5 movies most similar to "The Matrix"

conn = get_conn()
cur = conn.cursor()

# Get The Matrix's embedding
cur.execute("SELECT embedding FROM movies WHERE title = 'The Matrix'")
matrix_embedding = cur.fetchone()[0]

# Find 5 nearest neighbors using L2 distance (<->)
cur.execute("""
    SELECT title, genre, year, rating,
           embedding <-> %s::vector AS distance
    FROM movies
    WHERE title != 'The Matrix'
    ORDER BY embedding <-> %s::vector
    LIMIT 5
""", (matrix_embedding, matrix_embedding))

print("🔍 5 Movies Most Similar to 'The Matrix'")
print("=" * 60)
print(f"{'Title':<35} {'Genre':<12} {'Distance':>8}")
print("-" * 60)
for title, genre, year, rating, dist in cur.fetchall():
    print(f"{title:<35} {genre:<12} {dist:>8.4f}")

conn.close()

print("\n💡 The most similar movies are other Sci-Fi films!")
print("   This is because same-genre movies cluster together in vector space.")

---

## ❌ BAD: Linear Scan Through ALL Vectors

Right now, our similarity search works. But **how** does PostgreSQL find the nearest neighbors?

Without an index, it does a **sequential scan** — it computes the distance to EVERY SINGLE VECTOR in the table and then sorts. This is like searching for a book in a library by checking every single shelf.

With 40 movies this is instant. With **1 million** movies, this is a disaster.

Let's prove it with `EXPLAIN ANALYZE`:

In [ ]:
# Show that PostgreSQL uses a sequential scan (no index)

conn = get_conn()
cur = conn.cursor()

cur.execute("SELECT embedding FROM movies WHERE title = 'The Matrix'")
matrix_embedding = cur.fetchone()[0]

# EXPLAIN ANALYZE shows us HOW PostgreSQL executes the query
cur.execute("""
    EXPLAIN ANALYZE
    SELECT title, embedding <-> %s::vector AS distance
    FROM movies
    ORDER BY embedding <-> %s::vector
    LIMIT 5
""", (matrix_embedding, matrix_embedding))

print("📋 Query Plan (40 movies, NO index):")
print("=" * 70)
for row in cur.fetchall():
    print(f"  {row[0]}")

conn.close()

print()
print("⚠️  See 'Seq Scan' in the plan? That means PostgreSQL checked EVERY row.")
print("   With 40 movies this is fine. Let's see what happens at scale...")

### Scaling Up: Let's Feel the Pain

40 movies is too small to notice performance problems. Let's generate **50,000 random movie vectors** and see how linear scan performs:

In [ ]:
# Generate 50,000 random vectors to simulate a large dataset

conn = get_conn()
cur = conn.cursor()

# Create a benchmarking table
cur.execute("DROP TABLE IF EXISTS movies_large")
cur.execute("""
    CREATE TABLE movies_large (
        id SERIAL PRIMARY KEY,
        title VARCHAR(255),
        genre VARCHAR(100),
        embedding vector(128)
    )
""")

print("⏳ Inserting 50,000 vectors... ", end="", flush=True)
start = time.time()

# Generate vectors in batches for speed
genres = ['Sci-Fi', 'Action', 'Drama', 'Comedy', 'Horror', 'Romance', 'Animation', 'Thriller']
batch_size = 1000
for batch in range(50):
    values = []
    for i in range(batch_size):
        idx = batch * batch_size + i
        genre = genres[idx % len(genres)]
        # Random 128-dim vector
        vec = np.random.randn(128).tolist()
        vec_str = '[' + ','.join(f'{v:.6f}' for v in vec) + ']'
        values.append(f"('Movie {idx}', '{genre}', '{vec_str}')")

    cur.execute(f"INSERT INTO movies_large (title, genre, embedding) VALUES {','.join(values)}")

conn.commit()
elapsed = time.time() - start
print(f"done! ({elapsed:.1f}s)")

cur.execute("SELECT COUNT(*) FROM movies_large")
print(f"📊 Table now has {cur.fetchone()[0]:,} rows")
conn.close()

In [ ]:
# BAD: Linear scan on 50,000 vectors

conn = get_conn()
cur = conn.cursor()

# Create a query vector
query_vec = np.random.randn(128).tolist()
query_str = '[' + ','.join(f'{v:.6f}' for v in query_vec) + ']'

# Time the query (no index — sequential scan)
times = []
for _ in range(5):
    start = time.time()
    cur.execute("""
        SELECT title, embedding <-> %s::vector AS distance
        FROM movies_large
        ORDER BY embedding <-> %s::vector
        LIMIT 10
    """, (query_str, query_str))
    cur.fetchall()
    times.append((time.time() - start) * 1000)

avg_time = sum(times) / len(times)

# Show the query plan
cur.execute("""
    EXPLAIN ANALYZE
    SELECT title, embedding <-> %s::vector AS distance
    FROM movies_large
    ORDER BY embedding <-> %s::vector
    LIMIT 10
""", (query_str, query_str))

print("❌ BAD: Linear Scan on 50,000 Vectors")
print("=" * 60)
print(f"   Average query time: {avg_time:.1f} ms")
print(f"   Vectors scanned:   50,000 (ALL of them)")
print(f"   Recall:            100% (exact, but slow)")
print()
print("📋 Query Plan:")
for row in cur.fetchall():
    print(f"  {row[0]}")

conn.close()

print()
print("⚠️  ~{:.0f} ms for 50K vectors. Imagine 10M vectors — that's ~{:.0f} ms!".format(
    avg_time, avg_time * 200
))
print("   Linear scan is O(n). We need something better.")

---

## ✅ BETTER: IVFFlat Index

**IVFFlat** (Inverted File with Flat compression) works like organizing books into sections in a library:

1. **Build time**: K-means clustering divides all vectors into `lists` groups (like library sections)
2. **Query time**: Find which clusters are closest to the query, then only search those clusters

```
┌─────────────────────────────────────────┐
│           Vector Space (50K vectors)     │
│                                         │
│   ┌──────┐  ┌──────┐  ┌──────┐        │
│   │Clust │  │Clust │  │Clust │  ...    │
│   │  1   │  │  2   │  │  3   │        │
│   │ 500  │  │ 480  │  │ 520  │        │
│   │vectors│  │vectors│  │vectors│        │
│   └──────┘  └──────┘  └──────┘        │
│                                         │
│   Query: "Find similar to X"            │
│   → Check cluster 2 (closest center)    │
│   → Check cluster 3 (2nd closest)       │
│   → Skip all other clusters!            │
└─────────────────────────────────────────┘
```

**Key parameter**: `lists` = number of clusters. Rule of thumb: `sqrt(n)` where n = number of rows.
**Key parameter**: `probes` = how many clusters to search at query time. More probes = better recall, slower.

In [ ]:
# BETTER: Create an IVFFlat index

conn = get_conn()
cur = conn.cursor()

# Create IVFFlat index
# lists = number of clusters. sqrt(50000) ≈ 224, we'll use 100 for simplicity
print("⏳ Building IVFFlat index (100 lists)... ", end="", flush=True)
start = time.time()
cur.execute("""
    CREATE INDEX idx_movies_large_ivfflat
    ON movies_large
    USING ivfflat (embedding vector_l2_ops)
    WITH (lists = 100)
""")
conn.commit()
build_time = time.time() - start
print(f"done! ({build_time:.1f}s)")

# Query with IVFFlat (default probes = 1)
query_vec = np.random.randn(128).tolist()
query_str = '[' + ','.join(f'{v:.6f}' for v in query_vec) + ']'

# Set probes (how many clusters to search)
cur.execute("SET ivfflat.probes = 10")

times = []
for _ in range(5):
    start = time.time()
    cur.execute("""
        SELECT title, embedding <-> %s::vector AS distance
        FROM movies_large
        ORDER BY embedding <-> %s::vector
        LIMIT 10
    """, (query_str, query_str))
    cur.fetchall()
    times.append((time.time() - start) * 1000)

avg_ivf = sum(times) / len(times)

# Show query plan
cur.execute("""
    EXPLAIN ANALYZE
    SELECT title, embedding <-> %s::vector AS distance
    FROM movies_large
    ORDER BY embedding <-> %s::vector
    LIMIT 10
""", (query_str, query_str))

print()
print("✅ BETTER: IVFFlat Index (probes = 10)")
print("=" * 60)
print(f"   Build time:        {build_time:.1f}s")
print(f"   Average query time: {avg_ivf:.1f} ms")
print(f"   Clusters searched: 10 out of 100")
print()
print("📋 Query Plan:")
for row in cur.fetchall():
    print(f"  {row[0]}")

conn.close()

print()
print(f"🚀 Speedup vs linear scan: {avg_time/max(avg_ivf, 0.001):.1f}×")
print("   Instead of checking 50K vectors, we only checked ~5K (10 clusters).")

---

## 🏆 BEST: HNSW Index

**HNSW** (Hierarchical Navigable Small World) builds a **multi-layer graph** where each vector is a node connected to its neighbors. Think of it like a skip list for vectors:

```
Layer 2 (few nodes):     A -------- D -------- G
                         |          |          |
Layer 1 (more nodes):    A --- C -- D --- F -- G
                         |    |    |    |    |
Layer 0 (all nodes):     A-B-C-D-E-F-G-H-I-J-K
```

**Search**: Start at the top layer (coarse navigation), drop down layer by layer (finer navigation). Like finding a city → neighborhood → street → house.

**Key parameters**:
- `m` = max connections per node (default 16). Higher = better recall, more memory
- `ef_construction` = search width during build (default 64). Higher = better index quality, slower build
- `ef_search` = search width during query. Higher = better recall, slower query

HNSW typically gives **better recall than IVFFlat** at the same query speed.

In [ ]:
# BEST: Create an HNSW index

conn = get_conn()
cur = conn.cursor()

# Drop IVFFlat index first
cur.execute("DROP INDEX IF EXISTS idx_movies_large_ivfflat")
conn.commit()

# Create HNSW index
print("⏳ Building HNSW index (m=16, ef_construction=64)... ", end="", flush=True)
start = time.time()
cur.execute("""
    CREATE INDEX idx_movies_large_hnsw
    ON movies_large
    USING hnsw (embedding vector_l2_ops)
    WITH (m = 16, ef_construction = 64)
""")
conn.commit()
hnsw_build = time.time() - start
print(f"done! ({hnsw_build:.1f}s)")

# Query with HNSW
query_vec = np.random.randn(128).tolist()
query_str = '[' + ','.join(f'{v:.6f}' for v in query_vec) + ']'

# Set ef_search (higher = better recall, slower)
cur.execute("SET hnsw.ef_search = 40")

times = []
for _ in range(5):
    start = time.time()
    cur.execute("""
        SELECT title, embedding <-> %s::vector AS distance
        FROM movies_large
        ORDER BY embedding <-> %s::vector
        LIMIT 10
    """, (query_str, query_str))
    cur.fetchall()
    times.append((time.time() - start) * 1000)

avg_hnsw = sum(times) / len(times)

# Show query plan
cur.execute("""
    EXPLAIN ANALYZE
    SELECT title, embedding <-> %s::vector AS distance
    FROM movies_large
    ORDER BY embedding <-> %s::vector
    LIMIT 10
""", (query_str, query_str))

print()
print("🏆 BEST: HNSW Index (m=16, ef_construction=64, ef_search=40)")
print("=" * 60)
print(f"   Build time:         {hnsw_build:.1f}s")
print(f"   Average query time: {avg_hnsw:.1f} ms")
print()
print("📋 Query Plan:")
for row in cur.fetchall():
    print(f"  {row[0]}")

conn.close()

In [ ]:
# Side-by-side comparison

print("📊 BAD → BETTER → BEST Comparison (50,000 vectors)")
print("=" * 65)
print(f"{'Method':<25} {'Build Time':>12} {'Query Time':>12} {'Recall':>8}")
print("-" * 65)
print(f"{'❌ Linear Scan (BAD)':<25} {'N/A':>12} {f'{avg_time:.1f} ms':>12} {'100%':>8}")
print(f"{'✅ IVFFlat (BETTER)':<25} {f'{build_time:.1f}s':>12} {f'{avg_ivf:.1f} ms':>12} {'~90-95%':>8}")
print(f"{'🏆 HNSW (BEST)':<25} {f'{hnsw_build:.1f}s':>12} {f'{avg_hnsw:.1f} ms':>12} {'~95-99%':>8}")
print()
print("Key Takeaways:")
print("  • Linear scan guarantees 100% recall but doesn't scale")
print("  • IVFFlat is quick to build and good enough for many use cases")
print("  • HNSW is slower to build but gives the best query performance + recall")
print("  • Both ANN indexes trade tiny recall loss for massive speed gains")

## 🧹 Cleanup

In [ ]:
# Clean up the large benchmark table
conn = get_conn()
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS movies_large")
conn.commit()
conn.close()
print("🧹 Cleaned up benchmark table")

## 📚 Summary

### Key Takeaways

1. **Embeddings** are arrays of numbers where similar items have similar vectors
2. **Similarity metrics**: L2 distance (`<->`), cosine distance (`<=>`), inner product (`<#>`)
3. **Linear scan** checks every vector — O(n) and doesn't scale
4. **IVFFlat** partitions vectors into clusters — searches only relevant clusters
5. **HNSW** builds a navigable graph — best recall and query speed

### Interview Tip

> "For vector search, I'd use pgvector with an HNSW index. It gives ~99% recall with sub-millisecond queries. For datasets under 1M vectors, pgvector on Postgres is sufficient. Beyond that, I'd consider a dedicated vector database like Pinecone or Weaviate."

### Next Up

In **Notebook 2**, we'll benchmark IVFFlat vs HNSW in detail — build times, query speeds, recall at different scales, and how to tune parameters.